# Исследование коэффициента заражения β

**Авторы:** А. В. Королькова, PhD, Кулябов Д. С., DSc

**Принадлежность:** Российский университет дружбы народов

## Назначение скрипта

Данный скрипт исследует чувствительность модели SIR к изменению параметра β
(скорости заражения) при фиксированном коэффициенте выздоровления γ.

### Что делает скрипт

1. Для каждого значения β из заданного диапазона запускает детерминированную
   симуляцию модели SIR

2. Для каждого прогона вычисляет:
   - `peak_I` — максимальное число инфицированных (пик эпидемии)
   - `final_R` — конечное число выздоровевших

3. Сохраняет результаты в CSV-таблицу

4. Строит график зависимости `peak_I(β)` и `final_R(β)`

## Параметры исследования

| Параметр | Значение | Описание |
|----------|----------|----------|
| β_range | 0.1 : 0.05 : 0.8 | Диапазон коэффициента заражения (15 значений) |
| γ_fixed | 0.1 | Фиксированный коэффициент выздоровления |
| tmax | 100.0 | Время симуляции |
| S₀ | 990 | Начальное число восприимчивых |
| I₀ | 10 | Начальное число инфицированных |
| R₀ | 0 | Начальное число выздоровевших |

## Выходные данные

| Файл | Описание |
|------|----------|
| `data/sir_scan.csv` | Таблица с колонками: β, peak_I, final_R |
| `plots/sir_scan.png` | График зависимости peak_I(β) и final_R(β) — рис. 6.3 |

## Интерпретация результатов

- **При малых β (например, 0.1)** эпидемия не возникает:
  `peak_I ≈ 0`, `final_R ≈ 0`

- **С ростом β** пик заболеваемости сначала резко растёт,
  затем достигает насыщения (почти всё население переболевает)

- **Конечное R(β)** также растёт с β, но медленнее;
  при больших β практически всё население переходит в R

- **Пороговое явление:** график демонстрирует существование критического
  значения β, выше которого возникает вспышка эпидемии

## Инициализация проекта DrWatson

In [ ]:
using DrWatson
@quickactivate "project"

## Загрузка модуля SIRPetri

In [ ]:
include(srcdir("SIRPetri.jl"))
using .SIRPetri

## Подключение утилит для работы с данными и графикой

In [ ]:
using DataFrames, CSV, Plots

## Задание параметров сканирования

Диапазон β: от 0.1 до 0.8 с шагом 0.05

Фиксированный коэффициент выздоровления: γ = 0.1

Время симуляции: 100.0 единиц

In [ ]:
β_range = 0.1:0.05:0.8
γ_fixed = 0.1
tmax = 100.0

## Цикл по значениям β

Для каждого β выполняются следующие шаги:

1. Создание сети Петри с параметрами (β, γ_fixed)

2. Детерминированная симуляция (метод Tsit5, шаг сохранения 0.5)

3. Вычисление пика инфицированных `maximum(df.I)`

4. Вычисление конечного числа выздоровевших `df.R[end]`

5. Сохранение результатов в массив

In [ ]:
results = []

for β in β_range
    net, u0, _ = build_sir_network(β, γ_fixed)

    df = simulate_deterministic(net, u0, (0.0, tmax), saveat = 0.5, rates = [β, γ_fixed])

    peak_I = maximum(df.I)
    final_R = df.R[end]

    push!(results, (β = β, peak_I = peak_I, final_R = final_R))
end

## Сохранение результатов в CSV-файл

Таблица `sir_scan.csv` содержит три колонки:
- `β` — значение коэффициента заражения
- `peak_I` — пиковое число инфицированных
- `final_R` — конечное число выздоровевших

In [ ]:
df_scan = DataFrame(results)
CSV.write(datadir("sir_scan.csv"), df_scan)

## Визуализация результатов

### Рисунок 6.3: Зависимость peak_I(β) и final_R(β)

На графике отображаются две кривые:
- **Peak I** (синяя линия с маркерами) — пик заболеваемости
- **Final R** (красная линия с маркерами) — итоговое число переболевших

Характерные особенности графика:
- Пороговый эффект: при малых β эпидемия не развивается
- Резкий рост peak_I после критического значения β
- Насыщение peak_I и final_R при больших β

In [ ]:
p = plot(
    df_scan.β,
    [df_scan.peak_I df_scan.final_R],
    label = ["Peak I" "Final R"],
    marker = :circle,
    xlabel = "β (infection rate)",
    ylabel = "Population",
)

savefig(plotsdir("sir_scan.png"))

## Завершение работы

Скрипт успешно выполнил:
- Проведено 15 симуляций для β ∈ [0.1, 0.8]
- Сохранена таблица результатов → `data/sir_scan.csv`
- Построен график зависимости → `plots/sir_scan.png`

In [ ]:
println("Сканирование β завершено. Результат в data/sir_scan.csv")